# Quality benchmark — analysis

The generated document `docs/benchmarks/quality.md` holds the facts of the run. This notebook
is where the judgement goes: what the summary numbers are made of, where the models disagree
with each other and with the human readers, and what the photographs they get wrong look like.

Run it after `python -m benchmarks.quality`; everything here reads `results/quality/`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path('..').resolve() / 'results' / 'quality'

def per_image() -> pd.DataFrame:
    """Every model's answer about every photograph, one row each."""
    frames = []
    for path in sorted(RESULTS.glob('*/*.csv')):
        frame = pd.read_csv(path)
        frame['model'] = path.parent.name
        frame['unit'] = json.loads(path.with_suffix('.json').read_text())['unit']
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

def summaries() -> pd.DataFrame:
    rows = []
    for path in sorted(RESULTS.glob('*/*.json')):
        record = json.loads(path.read_text())
        summary = record['summary']
        rows.append({
            'model': record['model'], 'unit': record['unit'],
            'photographs': summary['photographs'], 'coverage': summary['coverage'],
            'accuracy': summary['gradeable']['accuracy'],
            'roc_auc': summary['gradeable']['roc_auc'],
            'kappa': summary['gradeable']['kappa'],
            'three_class': (summary['three_class'] or {}).get('accuracy'),
        })
    return pd.DataFrame(rows)

answers = per_image()
scores = summaries()
scores


## 1. Coverage first

A model that declines a photograph has not got it wrong. Coverage is the share of an
evaluation unit a model was willing to answer for at all, and it is read beside accuracy
rather than folded into it.


In [ ]:
outcomes = answers.pivot_table(
    index=['model', 'unit'], columns='outcome', values='key', aggfunc='count'
).fillna(0).astype(int)
outcomes


## 2. Accuracy and ranking are different questions

Accuracy depends on where a model's threshold sits, and the toolbox's authors say plainly that
theirs does not transfer between datasets. The area under the ROC curve does not depend on a
threshold. A model can therefore rank photographs almost perfectly and still be called
inaccurate — which is a statement about the threshold, not about the model.


In [ ]:
wide = scores.pivot(index='unit', columns='model', values=['accuracy', 'roc_auc'])
figure, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for axis, metric in zip(axes, ['accuracy', 'roc_auc']):
    wide[metric].plot.barh(ax=axis, xlim=(0.4, 1.0))
    axis.set_title(metric)
    axis.legend(fontsize=7)
figure.tight_layout()


### 2.1 The whole ROC curve, not one point on it

Each curve is one model on one unit. Where two curves cross, the better model depends on
whether you would rather miss a bad photograph or throw away a good one.


In [ ]:
def curve(frame):
    """The true and false positive rates at every threshold the model actually produced."""
    worth = frame['grade'].isin(['good', 'usable']).to_numpy()
    score = frame['gradeable'].to_numpy()
    order = np.argsort(-score)
    worth = worth[order]
    true = np.cumsum(worth) / max(worth.sum(), 1)
    false = np.cumsum(~worth) / max((~worth).sum(), 1)
    return np.r_[0, false], np.r_[0, true]

graded = answers[answers['outcome'] == 'graded']
units = sorted(graded['unit'].unique())
figure, axes = plt.subplots(1, len(units), figsize=(4 * len(units), 4), squeeze=False)
for axis, unit in zip(axes[0], units):
    for model, frame in graded[graded['unit'] == unit].groupby('model'):
        axis.plot(*curve(frame), label=model, linewidth=1)
    axis.plot([0, 1], [0, 1], color='grey', linewidth=0.5)
    axis.set_title(unit, fontsize=9)
    axis.set_xlabel('kept, though bad')
    axis.set_ylabel('kept, and worth measuring')
    axis.legend(fontsize=6)
figure.tight_layout()


## 3. Where the models disagree with each other

Two models agreeing is not two pieces of evidence when they learned from the same labels.
[QuickQual](../docs/models/quickqual.md) and the
[AutoMorph grader](../docs/models/automorph-quality-grader.md) were both fitted on EyeQ, so
their agreement is expected; agreement with the toolbox ensemble, trained on other data, is
the more informative number.


In [ ]:
verdicts = graded.assign(worth=lambda f: f['gradeable'] >= 0.5).pivot_table(
    index=['unit', 'key'], columns='model', values='worth'
)
models = list(verdicts.columns)
agreement = pd.DataFrame(
    [[(verdicts[a] == verdicts[b]).mean() for b in models] for a in models],
    index=models, columns=models,
)
agreement.round(3)


## 4. Where the models disagree with the readers

Two of these datasets kept their readers apart, so the photographs the readers themselves
disagreed about can be separated from the ones they were sure of. A model that is wrong where
the humans disagreed is in different trouble from one that is wrong where they did not.


In [ ]:
def readers(entry: str) -> list[str]:
    return [part.split('=')[1] for part in entry.split(';') if '=' in part]

multi = graded[graded['readers'].fillna('') != ''].copy()
multi['unanimous'] = multi['readers'].map(lambda entry: len(set(readers(entry))) == 1)
multi['right'] = (multi['gradeable'] >= 0.5) == multi['grade'].isin(['good', 'usable'])
multi.pivot_table(index=['model', 'unit'], columns='unanimous', values='right').round(3)


## 5. The photographs everything gets wrong

A table says a model scores 0.82. This is where someone finds out whether the 0.82 is two
populations, and whether one of them is a camera.


In [ ]:
from PIL import Image

STORE = Path('..').resolve() / '.atlas_data'

wrong = graded.assign(
    right=lambda f: (f['gradeable'] >= 0.5) == f['grade'].isin(['good', 'usable'])
).groupby(['unit', 'key'])['right'].mean()
hardest = wrong[wrong == 0].index[:8]

figure, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for axis, (unit, key) in zip(axes.ravel(), hardest):
    slug = unit.split('/')[0]
    axis.imshow(Image.open(STORE / slug / '512' / 'images' / f'{key}.png'))
    said = graded[(graded['unit'] == unit) & (graded['key'] == key)]
    axis.set_title(f"{key}\ndataset says {said['grade'].iloc[0]}", fontsize=8)
    axis.axis('off')
figure.tight_layout()


## 6. What this benchmark cannot say

- **A model marked `unknown` is not cleared.** VascX publishes no list of what it trained on,
  and its shipped configuration names EyeQ for this model, so an out-of-sample claim cannot be
  made for it here.
- **The three-class comparison is not like for like.** The datasets' own three grades were
  defined by different people for different purposes: FQS's come from a mean opinion score,
  MSHF's from three annotators scoring clarity, contrast and illumination, and FIVES's are
  derived from its own defect labels.
- **Nothing here says a model is best.** Read a row, not a column.
